In [ ]:
import hail as hl
# Initialize Hail 
hl.init(default_reference = 'GRCh38')

In [3]:
mt = hl.read_matrix_table("gs://ibd-exomes-gnomad-subset/QC_round2/1.split_sites/ibd_exomes_sites_splitted.mt")
mt.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
    'info': struct {
        AC_raw: array<int32>, 
        AN_raw: int32, 
        AF_raw: array<float64>, 
        nhomalt_raw: array<int32>, 
        AC: array<int32>, 
        AN: int32, 
        AF: array<float64>, 
        nhomalt: array<int32>
    }
    'a_index': int32
    'was_split': bool
----------------------------------------
Entry fields:
    'GT': call
    'DP': int32
    'GQ': int32
    'RGQ': int32
    'gvcf_info': struct {
        AC: array<int32>, 
        AF: array<float64>, 
        AN: int32, 
        AS_BaseQRankSum: array<float64>, 
        AS_FS: array<float64>, 
        AS_InbreedingCoeff: array<float64>, 
        AS_MQ: array<float64>, 
        AS_MQRankSum: array<float64>, 
        AS_QD: array<float64>, 
 

In [5]:
rg37 = hl.get_reference('GRCh37')
rg38 = hl.get_reference('GRCh38')
rg37.add_liftover('gs://hail-common/references/grch37_to_grch38.over.chain.gz', rg38) 

In [4]:
#mt = mt.filter_rows(mt.alleles[1] == "*", keep = False)  
#mt.show(10)
mt_removed = mt.filter_rows(mt.alleles[1] == "*", keep = True) #checks if the alternate allele (the second element in the alleles array) is *. 
                                                               # The * symbol typically represents a spanning deletion.
mt_removed.show(10)

,
locus,alleles
locus<GRCh38>,array<str>
chr1:14937,"[""T"",""*""]"
chr1:15811,"[""T"",""*""]"
chr1:15817,"[""G"",""*""]"
chr1:15820,"[""G"",""*""]"
chr1:17361,"[""T"",""*""]"
chr1:17502,"[""T"",""*""]"
chr1:17556,"[""C"",""*""]"
chr1:30923,"[""G"",""*""]"


In [6]:
# Get NFE AF
nfe_ht = hl.read_table('gs://daly-finland-konrad/sum_stats_enrichments/fin_enriched_exomes_37.ht/')
nfe_ht = nfe_ht.naive_coalesce(1000) # Coalesce Partitions: reducing the number of partitions of a dataset
                                     # Partitions are subsets of the data that can be processed in parallel across multiple nodes in a cluster. 

In [ ]:
nfe_ht = nfe_ht.annotate(new_locus = hl.liftover(nfe_ht.locus, 'GRCh38'))
nfe_ht = nfe_ht.filter(hl.is_defined(nfe_ht.new_locus))
nfe_ht = nfe_ht.key_by(locus = nfe_ht.new_locus, alleles = nfe_ht.alleles)
mt = mt.annotate_rows(nfe_ht = nfe_ht[mt.row_key])

vars_of_interest = hl.literal({"frameshift_variant", "inframe_deletion", "inframe_insertion", "stop_lost", "stop_gained", "start_lost", "splice_acceptor_variant", "splice_donor_variant", "splice_region_variant", "missense_variant", "synonymous_variant"})

variants = mt.rows()
print("Annotating with VEP...")
variants = hl.methods.vep(variants, "gs://hail-us-vep/vep95-GRCh38-loftee-gcloud.json")

#Filter for all variants that have a defined most severe consequence that is part of the literal containing all variant types of interest
print("Filtering for variants of interest...")
variants = variants.filter(hl.is_defined(variants.vep.most_severe_consequence))
variants = variants.annotate_globals(x = vars_of_interest)
variants = variants.filter(variants.x.contains(variants.vep.most_severe_consequence))

print("Variants left:")
print(variants.count())

#Annotate back in
mt = mt.annotate_rows(vep = variants[mt.row_key].vep)
mt = mt.filter_rows(hl.is_defined(mt.vep))

In [ ]:
mt = mt.write("gs://ibd-exomes-gnomad-subset/QC_round2/2.variant_narrowing/variant_narrowed.mt", overwrite=True)

In [ ]:
mt.count()